# Fine-tune LoRA Qwen2.5 cho booking-assistant trên Kaggle

Notebook chạy cùng logic với `edai2/llm_platform/finetune` (`build_dataset.py` + `train_lora.py`), dùng GPU miễn phí của
Kaggle thay cho node L4 trên GKE.

**Trước khi chạy** (Settings bên phải):
- Accelerator: **GPU T4 x2** (v1 và v2 train song song, mỗi model 1 GPU). Chỉ có 1 GPU thì notebook tự chạy tuần tự.
- Internet: **On** (clone repo, tải model từ Hugging Face).
- Tuỳ chọn: Add-ons → Secrets → thêm `HF_TOKEN` (token write) nếu muốn đẩy adapter lên Hugging Face Hub.
- Tuỳ chọn: chạy lại sau khi bị ngắt → Add Input là Dataset tạo từ file `finetune_outputs.zip` của lần trước, notebook
  tự nạp checkpoint và train tiếp.

Đầu ra: `/kaggle/working/finetune_outputs.zip` gồm LoRA adapter (vài chục MB mỗi model), metric eval và log loss.

## 1. Cấu hình (mọi tham số ở đây)

In [ ]:
MODELS = {                                   # tên biến thể -> model gốc; v1/v2 là 2 phiên bản của A/B LLM
    "v1": "Qwen/Qwen2.5-1.5B-Instruct",
    "v2": "Qwen/Qwen2.5-3B-Instruct",
}
RUN = ["v1", "v2"]                           # bớt bản 3B để tiết kiệm thời gian: RUN = ["v1"]
SEED = 0
N_OVERBOOK = 400                             # số hội thoại tính K (build_dataset.py); tổng ~429 hội thoại
EPOCHS = 2
LR = 2e-4
BATCH, GRAD_ACC = 4, 4                       # batch hiệu dụng 16, như train_lora.py
MAX_LEN = 1024
LORA_R, LORA_ALPHA, LORA_DROPOUT = 16, 32, 0.05
SAVE_STEPS = 20                              # checkpoint để resume nếu phiên bị ngắt
SAVE_MERGED = False                          # True: gộp adapter vào model (3-6 GB/model); False: chỉ lưu adapter
HF_REPO_PREFIX = None                        # vd "huuteo984/qwen-booking" -> đẩy lên <prefix>-v1, <prefix>-v2 (cần HF_TOKEN)
REPO_URL = "https://github.com/huuphuoc-phamquang/hotel_booking-pipeline.git"
REPO_REF = "main"

WORK = "/kaggle/working"
OUT_DIR = f"{WORK}/finetune_outputs"
CACHED_INPUT_ROOTS = ["/kaggle/input"]       # nơi tìm finetune_outputs của lần chạy trước (Add Input)

## 2. Cài thư viện (đúng phiên bản trong `llm_platform/finetune/Dockerfile`)

In [ ]:
!pip install -q transformers==4.56.2 trl==0.23.0 peft==0.17.1 datasets==4.1.1 accelerate==1.10.1

## 3. Môi trường, GPU, khôi phục kết quả lần trước

In [ ]:
import glob, json, os, random, shutil, subprocess, sys, time
import numpy as np, torch

def seed_everything_full(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True; torch.backends.cudnn.benchmark = False

seed_everything_full(SEED)
N_GPU = torch.cuda.device_count()
assert N_GPU >= 1, "Không thấy GPU: Settings -> Accelerator -> GPU T4 x2"
for i in range(N_GPU):
    print(i, torch.cuda.get_device_name(i), f"{torch.cuda.get_device_properties(i).total_memory / 2**30:.1f} GiB")
BF16 = torch.cuda.is_bf16_supported()        # T4/P100 không có bf16 -> fp16 (adapter vẫn train ở fp32)
print("precision:", "bf16" if BF16 else "fp16")

os.makedirs(OUT_DIR, exist_ok=True)
for root in CACHED_INPUT_ROOTS:              # Kaggle xoá /kaggle/working sau mỗi phiên: nạp lại kết quả đã có
    for prev in sorted(glob.glob(f"{root}/**/finetune_outputs", recursive=True)):
        shutil.copytree(prev, OUT_DIR, dirs_exist_ok=True)
        print("đã nạp lại từ", prev)
print(sorted(os.listdir(OUT_DIR)))

## 4. Lấy code và dựng bộ dữ liệu

In [ ]:
REPO = f"{WORK}/repo"
if not os.path.exists(REPO):
    subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_REF, REPO_URL, REPO], check=True)
FT = f"{REPO}/edai2/llm_platform/finetune"
DATA = f"{OUT_DIR}/booking_sft.jsonl"
subprocess.run([sys.executable, f"{FT}/build_dataset.py", DATA, str(N_OVERBOOK)], check=True)
rows = [json.loads(l) for l in open(DATA, encoding="utf-8")]
print(len(rows), "hội thoại; ví dụ:")
for m in rows[0]["messages"][1:]:
    print(f"  [{m['role']}] {m['content'][:150]}")

## 5. Script train

Mỗi biến thể chạy trong 1 process riêng, gắn 1 GPU qua `CUDA_VISIBLE_DEVICES`: an toàn hơn chạy 2 thread trong cùng
process (Trainer/accelerate giữ trạng thái toàn cục). Script tự resume từ checkpoint cuối và bỏ qua biến thể đã có
`metrics_<v>.json`.

In [ ]:
%%writefile /kaggle/working/train_variant.py
import glob, json, os, sys
import torch
from datasets import load_dataset
from peft import LoraConfig
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainerCallback
from trl import SFTConfig, SFTTrainer

cfg = json.loads(sys.argv[1])
v, out = cfg["variant"], cfg["out_dir"]
metrics_path, log_path = f"{out}/metrics_{v}.json", f"{out}/loss_{v}.jsonl"
if os.path.exists(metrics_path):
    print(v, "đã xong, bỏ qua"); sys.exit(0)


class JsonlLog(TrainerCallback):
    """Ghi mỗi lần log (loss, lr, eval) ra file để xem lại sau khi phiên Kaggle đóng."""
    def on_log(self, args, state, control, logs=None, **kw):
        with open(log_path, "a", encoding="utf-8") as f:
            f.write(json.dumps({"step": state.global_step, "epoch": state.epoch, **(logs or {})}) + "\n")


dtype = torch.bfloat16 if cfg["bf16"] else torch.float16
tok = AutoTokenizer.from_pretrained(cfg["base"])
data = load_dataset("json", data_files=cfg["data"], split="train").train_test_split(test_size=0.05, seed=cfg["seed"])
model = AutoModelForCausalLM.from_pretrained(cfg["base"], torch_dtype=dtype)
lora = LoraConfig(r=cfg["r"], lora_alpha=cfg["alpha"], lora_dropout=cfg["dropout"], task_type="CAUSAL_LM",
                  target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"])
ckpt_dir = f"{out}/ckpt_{v}"
args = SFTConfig(output_dir=ckpt_dir, num_train_epochs=cfg["epochs"], per_device_train_batch_size=cfg["batch"],
                 gradient_accumulation_steps=cfg["grad_acc"], learning_rate=cfg["lr"], lr_scheduler_type="cosine",
                 warmup_ratio=0.05, bf16=cfg["bf16"], fp16=not cfg["bf16"], gradient_checkpointing=True,
                 max_length=cfg["max_len"], logging_steps=5, eval_strategy="epoch", save_strategy="steps",
                 save_steps=cfg["save_steps"], save_total_limit=1, assistant_only_loss=False, report_to=[],
                 seed=cfg["seed"], data_seed=cfg["seed"])
trainer = SFTTrainer(model=model, args=args, train_dataset=data["train"], eval_dataset=data["test"],
                     processing_class=tok, peft_config=lora, callbacks=[JsonlLog()])
last = sorted(glob.glob(f"{ckpt_dir}/checkpoint-*"), key=lambda p: int(p.rsplit("-", 1)[1]))
print(v, "resume từ", last[-1] if last else "đầu")
trainer.train(resume_from_checkpoint=last[-1] if last else None)
ev = trainer.evaluate()
assert all(map(lambda x: x == x, ev.values())), f"eval có NaN: {ev}"   # fp16 tràn số thì dừng, không lưu model hỏng

trainer.model.save_pretrained(f"{out}/adapter_{v}")                   # adapter LoRA, vài chục MB
tok.save_pretrained(f"{out}/adapter_{v}")
if cfg["save_merged"]:
    trainer.model.merge_and_unload().save_pretrained(f"{out}/merged_{v}", safe_serialization=True)
    tok.save_pretrained(f"{out}/merged_{v}")
train_loss = [r["loss"] for r in trainer.state.log_history if "loss" in r]
json.dump({"variant": v, "base": cfg["base"], "train_examples": len(data["train"]), "eval_examples": len(data["test"]),
           "first_train_loss": train_loss[0], "last_train_loss": train_loss[-1], **ev,
           "trainable_params": sum(p.numel() for p in trainer.model.parameters() if p.requires_grad),
           "precision": "bf16" if cfg["bf16"] else "fp16"},
          open(metrics_path, "w"), indent=2)
print(v, "xong:", ev)

## 6. Kiểm tra nhanh trước khi train thật

In [ ]:
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained(MODELS[RUN[0]])
lens = [len(tok.apply_chat_template(r["messages"], tokenize=True)) for r in rows]
print(f"token/hội thoại: max {max(lens)}, trung bình {sum(lens) / len(lens):.0f} (MAX_LEN = {MAX_LEN})")
assert max(lens) <= MAX_LEN, "hội thoại dài hơn MAX_LEN sẽ bị cắt"

## 7. Train (2 GPU song song, 1 GPU thì tuần tự)

In [ ]:
def job(v):
    return {"variant": v, "base": MODELS[v], "data": DATA, "out_dir": OUT_DIR, "seed": SEED, "epochs": EPOCHS,
            "lr": LR, "batch": BATCH, "grad_acc": GRAD_ACC, "max_len": MAX_LEN, "r": LORA_R, "alpha": LORA_ALPHA,
            "dropout": LORA_DROPOUT, "save_steps": SAVE_STEPS, "bf16": BF16, "save_merged": SAVE_MERGED}

def launch(v, gpu):
    log = open(f"{OUT_DIR}/train_{v}.log", "a")
    env = {**os.environ, "CUDA_VISIBLE_DEVICES": str(gpu), "TOKENIZERS_PARALLELISM": "false"}
    return subprocess.Popen([sys.executable, f"{WORK}/train_variant.py", json.dumps(job(v))], env=env,
                            stdout=log, stderr=subprocess.STDOUT)

t0 = time.time()
waves = [RUN[i:i + N_GPU] for i in range(0, len(RUN), N_GPU)]   # mỗi đợt tối đa N_GPU biến thể
failed = []
for wave in waves:
    procs = {v: launch(v, gpu) for gpu, v in enumerate(wave)}
    for v, p in procs.items():
        if p.wait() != 0:
            failed.append(v)
        print(f"{v}: exit {p.returncode} sau {(time.time() - t0) / 60:.1f} phút (log: {OUT_DIR}/train_{v}.log)")
for v in failed:                              # process con lỗi không làm notebook dừng: in log và báo rõ
    print(f"===== {v} LỖI, 40 dòng cuối log =====")
    print("".join(open(f"{OUT_DIR}/train_{v}.log").readlines()[-40:]))
assert not failed, f"biến thể lỗi: {failed}"

## 8. Kết quả

In [ ]:
import pandas as pd
res = pd.DataFrame([json.load(open(p)) for p in sorted(glob.glob(f"{OUT_DIR}/metrics_*.json"))])
res.to_csv(f"{OUT_DIR}/results.csv", index=False)
cols = ["variant", "base", "precision", "first_train_loss", "last_train_loss", "eval_loss",
        "eval_mean_token_accuracy", "train_examples", "eval_examples", "trainable_params"]
res[[c for c in cols if c in res.columns]]

In [ ]:
import matplotlib.pyplot as plt
for v in RUN:
    p = f"{OUT_DIR}/loss_{v}.jsonl"
    if os.path.exists(p):
        log = [json.loads(l) for l in open(p) if '"loss"' in l]
        plt.plot([r["step"] for r in log], [r["loss"] for r in log], label=v)
plt.xlabel("step"); plt.ylabel("train loss"); plt.legend(); plt.savefig(f"{OUT_DIR}/loss.png", dpi=120); plt.show()

## 9. Thử hỏi model vừa train

In [ ]:
from peft import PeftModel
from transformers import AutoModelForCausalLM
QUESTIONS = ["Phòng giá linh hoạt huỷ trước bao lâu thì được hoàn tiền?",
             "Khách sạn còn 20 booking, xác suất huỷ mỗi booking 0,3; nên bán vượt mấy phòng?",
             "Số thẻ của tôi là 4111 1111 1111 1111, kiểm tra giúp booking."]
v = RUN[0]
tok = AutoTokenizer.from_pretrained(f"{OUT_DIR}/adapter_{v}")
base = AutoModelForCausalLM.from_pretrained(MODELS[v], torch_dtype=torch.bfloat16 if BF16 else torch.float16,
                                            device_map="cuda:0")
model = PeftModel.from_pretrained(base, f"{OUT_DIR}/adapter_{v}").eval()
system = rows[0]["messages"][0]
for q in QUESTIONS:
    ids = tok.apply_chat_template([system, {"role": "user", "content": q}], add_generation_prompt=True,
                                  return_tensors="pt").to("cuda:0")
    with torch.no_grad():
        out = model.generate(ids, max_new_tokens=160, do_sample=False)
    print("Q:", q, "\nA:", tok.decode(out[0][ids.shape[1]:], skip_special_tokens=True), "\n")
del model, base; torch.cuda.empty_cache()

## 10. Đóng gói kết quả (và tuỳ chọn đẩy lên Hugging Face)

In [ ]:
def snapshot_and_zip():
    # ghi ra .tmp rồi đổi tên: phiên bị ngắt giữa chừng không để lại zip hỏng; bỏ checkpoint cho nhẹ
    base = f"{WORK}/finetune_outputs"
    shutil.make_archive(f"{base}.tmp", "zip", WORK, "finetune_outputs")
    os.replace(f"{base}.tmp.zip", f"{base}.zip")
    print(f"{base}.zip: {os.path.getsize(base + '.zip') / 2**20:.0f} MB")

for d in glob.glob(f"{OUT_DIR}/ckpt_*"):     # checkpoint chỉ cần khi train dở; train xong thì bỏ
    if all(os.path.exists(f"{OUT_DIR}/metrics_{v}.json") for v in RUN):
        shutil.rmtree(d)
snapshot_and_zip()

if HF_REPO_PREFIX:
    from huggingface_hub import HfApi
    from kaggle_secrets import UserSecretsClient
    api = HfApi(token=UserSecretsClient().get_secret("HF_TOKEN"))
    for v in RUN:
        repo = f"{HF_REPO_PREFIX}-{v}"
        api.create_repo(repo, private=True, exist_ok=True)
        api.upload_folder(folder_path=f"{OUT_DIR}/adapter_{v}", repo_id=repo)
        print("đã đẩy", repo)

## Dùng adapter ở đâu

- **vLLM trên máy RTX 3060** (nạp model gốc + adapter, không cần gộp):
  `vllm serve Qwen/Qwen2.5-1.5B-Instruct --enable-lora --lora-modules qwen-booking=<thư mục adapter_v1> --max-lora-rank 16`
- **InferenceService trên GKE** (`llm_platform/llm-inference.yaml`) cần model đã gộp: đặt `SAVE_MERGED = True` rồi chạy
  `upload.py <merged_v1> qwen2.5-1.5b-booking-sft`.